In [0]:
#retomando setup
from pyspark.sql import functions as F
import pandas as pd
import warnings
import warnings
warnings.filterwarnings("ignore")
CATALOGO, SCHEMA = "workspace", "default"


promovendo a silver_turbina para camada gold, pois já saiu no formato da dimensão

In [0]:
dim_turbina = spark.table(f"{CATALOGO}.{SCHEMA}.silver_turbina")

(dim_turbina.write.mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable(f"{CATALOGO}.{SCHEMA}.gold_dim_turbina"))

display(spark.table(f"{CATALOGO}.{SCHEMA}.gold_dim_turbina").orderBy("turbina_id"))

turbina_id,nome,latitude,longitude,potencia_nominal_kw,diametro_rotor_m,altura_cubo_m
1,Kelmarsh 1,52.400604,-0.947133,2050,92.0,78.5
2,Kelmarsh 2,52.402551,-0.949527,2050,92.0,78.5
3,Kelmarsh 3,52.403834,-0.94419,2050,92.0,68.5
4,Kelmarsh 4,52.398781,-0.94115,2050,92.0,78.5
5,Kelmarsh 5,52.402308,-0.940537,2050,92.0,78.5
6,Kelmarsh 6,52.400687,-0.936093,2050,92.0,68.5


próximo passo é explodir o timestamp para possibilar perguntas em granularidades diferentes

In [0]:
scada = spark.table(f"{CATALOGO}.{SCHEMA}.silver_scada")

#separando instantes distintos
tempos = scada.select("tempo").distinct()

#quebra cada instante nos seus componentes
dim_tempo = tempos.select(
    F.col("tempo").alias("tempo_id"),
    F.to_date("tempo").alias("data"),
    F.year("tempo").alias("ano"),
    F.month("tempo").alias("mes"),
    F.dayofmonth("tempo").alias("dia"),
    F.hour("tempo").alias("hora"),
    F.minute("tempo").alias("minuto"),
    F.date_format("tempo","EEEE").alias("dia_semana")
)

(dim_tempo.write.mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable(f"{CATALOGO}.{SCHEMA}.gold_dim_tempo"))

display(spark.table(f"{CATALOGO}.{SCHEMA}.gold_dim_tempo").orderBy("tempo_id"))


tempo_id,data,ano,mes,dia,hora,minuto,dia_semana
2018-01-01T00:00:00.000Z,2018-01-01,2018,1,1,0,0,Monday
2018-01-01T00:10:00.000Z,2018-01-01,2018,1,1,0,10,Monday
2018-01-01T00:20:00.000Z,2018-01-01,2018,1,1,0,20,Monday
2018-01-01T00:30:00.000Z,2018-01-01,2018,1,1,0,30,Monday
2018-01-01T00:40:00.000Z,2018-01-01,2018,1,1,0,40,Monday
2018-01-01T00:50:00.000Z,2018-01-01,2018,1,1,0,50,Monday
2018-01-01T01:00:00.000Z,2018-01-01,2018,1,1,1,0,Monday
2018-01-01T01:10:00.000Z,2018-01-01,2018,1,1,1,10,Monday
2018-01-01T01:20:00.000Z,2018-01-01,2018,1,1,1,20,Monday
2018-01-01T01:30:00.000Z,2018-01-01,2018,1,1,1,30,Monday


In [0]:
print("dim_turbina:")
display(spark.table(f"{CATALOGO}.{SCHEMA}.gold_dim_turbina").orderBy("turbina_id"))

print("dim_tempo (contagem — deve bater com os instantes distintos do SCADA):")
print(spark.table(f"{CATALOGO}.{SCHEMA}.gold_dim_tempo").count())

dim_turbina:


turbina_id,nome,latitude,longitude,potencia_nominal_kw,diametro_rotor_m,altura_cubo_m
1,Kelmarsh 1,52.400604,-0.947133,2050,92.0,78.5
2,Kelmarsh 2,52.402551,-0.949527,2050,92.0,78.5
3,Kelmarsh 3,52.403834,-0.94419,2050,92.0,68.5
4,Kelmarsh 4,52.398781,-0.94115,2050,92.0,78.5
5,Kelmarsh 5,52.402308,-0.940537,2050,92.0,78.5
6,Kelmarsh 6,52.400687,-0.936093,2050,92.0,68.5


dim_tempo (contagem — deve bater com os instantes distintos do SCADA):
52560


fazendo one-hot encoding para definir as operaçoes normais e com parada (status de funcionamento)

In [0]:
dados = [(0, "Normal", "operacao sem evento de parada"),
         (1, "Stop","turbina parada (evento de duracao)")]

dim_status = spark.createDataFrame(dados, ["status_id", "status", "descricao"])

(dim_status.write.mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable(f"{CATALOGO}.{SCHEMA}.gold_dim_status"))

display(spark.table(f"{CATALOGO}.{SCHEMA}.gold_dim_status"))

status_id,status,descricao
0,Normal,operacao sem evento de parada
1,Stop,turbina parada (evento de duracao)


## JOIN PARA RECONCILIAR

In [0]:
scada = spark.table(f"{CATALOGO}.{SCHEMA}.silver_scada")
status = spark.table(f"{CATALOGO}.{SCHEMA}.silver_status")

#primeiro os eventos de parada com janela completa (inicio e fim)
paradas = status.filter(
    (F.col("status") == "Stop") &
    F.col("tempo_inicio").isNotNull() &
    F.col("tempo_fim").isNotNull()
 ).select(  #primeiro filtro só os stops e depois faço um select
     F.col("turbina_id").alias("s_turbina"),
     "tempo_inicio",
     "tempo_fim"
 ) 

#left join por faixa (uso o left para não perder as leituras normais e tipar todo o scada)

juncao = scada.join(
    paradas,
    on=(
        (scada.turbina_id == paradas.s_turbina) &
        (scada.tempo >= paradas.tempo_inicio) &
        (scada.tempo < paradas.tempo_fim)
    ),
    how = "left"
)

fato = juncao.withColumn(
    "status_id",
    F.when(F.col("tempo_inicio").isNotNull(), 1).otherwise(0)
)


selecionando o que apenas o gold_fato precisa (as 13 features selecionadas previamente nas outras camadas)

In [0]:
gold_fato = fato.select(
    "turbina_id",
    F.col("tempo").alias("tempo_id"),
    "status_id",
    "wind_speed_m_s",
    "wind_speed_std_m_s",
    "wind_speed_min_m_s",
    "wind_speed_max_m_s",
    "power_kw",
    "rotor_speed_rpm",
    "generator_rpm",
    "blade_pitch_a_deg",
    "front_bearing_temp_c",
    "rear_bearing_temp_c",
    "gear_oil_temp_c",
    "generator_bearing_front_temp_c",
    "nacelle_ambient_temp_c"
)

(gold_fato.write.mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable(f"{CATALOGO}.{SCHEMA}.gold_fato_scada")
)

print("gold_fato_scada criado")

gold_fato_scada criado


validando a reconciliação (join)

In [0]:
fato = spark.table(f"{CATALOGO}.{SCHEMA}.gold_fato_scada")

#esperado retornar aproximadamente 315.360 registros, que são correspondentes a 52.600 registros por turbina
print("total de linhas:", fato.count())

#distribuicao do carimbo normal por stop
display(fato.groupBy("status_id").count().orderBy("status_id"))

total de linhas: 315360


status_id,count
0,302071
1,13289


numeros que fazem sentido com o esperado de engenharia, cerca de 96% do tmepo operando normal e 4% em parada. o tratamento foi limpo e os resultados estão coerentes com a realidade. Principalmente porque esses 4% estão conversando com os 3,5% dos valores nulos lidos na potencia elétrica gerada (power_kw)

# Metadados
Atualizando metadados para por no Unit Catalog

In [0]:
%sql
-- ===== TABELAS =====
COMMENT ON TABLE workspace.default.gold_fato_scada IS
  'Fato SCADA do parque eólico de Kelmarsh (2018). Grão: 1 linha por turbina x instante de 10 min. status_id reconciliado com eventos Stop.';
COMMENT ON TABLE workspace.default.gold_fato_scada_curado IS
  'Versão curada do gold_fato_scada: generator_rpm negativo (ruído de sensor) e rotor fora de [0,16] rpm convertidos em nulo. Base das análises.';
COMMENT ON TABLE workspace.default.gold_dim_turbina IS
  'Dimensão turbina: cadastro das 6 turbinas Senvion MM92. Fonte: Kelmarsh_WT_static.csv.';
COMMENT ON TABLE workspace.default.gold_dim_tempo IS
  'Dimensão tempo: 52.560 instantes de 10 min de 2018, em UTC.';
COMMENT ON TABLE workspace.default.gold_dim_status IS
  'Dimensão status operacional: 0 = Normal, 1 = Stop.';

-- ===== FATO =====
ALTER TABLE workspace.default.gold_fato_scada ALTER COLUMN turbina_id COMMENT 'FK -> gold_dim_turbina';
ALTER TABLE workspace.default.gold_fato_scada ALTER COLUMN tempo_id COMMENT 'FK -> gold_dim_tempo. Instante da leitura (UTC)';
ALTER TABLE workspace.default.gold_fato_scada ALTER COLUMN status_id COMMENT 'FK -> gold_dim_status. 0 Normal, 1 Stop (range join com silver_status)';
ALTER TABLE workspace.default.gold_fato_scada ALTER COLUMN wind_speed_m_s COMMENT 'Velocidade média do vento (m/s)';
ALTER TABLE workspace.default.gold_fato_scada ALTER COLUMN wind_speed_std_m_s COMMENT 'Desvio-padrão do vento no intervalo (m/s)';
ALTER TABLE workspace.default.gold_fato_scada ALTER COLUMN wind_speed_min_m_s COMMENT 'Vento mínimo no intervalo (m/s)';
ALTER TABLE workspace.default.gold_fato_scada ALTER COLUMN wind_speed_max_m_s COMMENT 'Vento máximo no intervalo (m/s)';
ALTER TABLE workspace.default.gold_fato_scada ALTER COLUMN power_kw COMMENT 'Potência ativa média (kW). Nominal 2050 kW';
ALTER TABLE workspace.default.gold_fato_scada ALTER COLUMN rotor_speed_rpm COMMENT 'Rotação do rotor (rpm). Faixa do fabricante 7,8-15';
ALTER TABLE workspace.default.gold_fato_scada ALTER COLUMN generator_rpm COMMENT 'Rotação do gerador (rpm). Faixa do fabricante 900-1800';
ALTER TABLE workspace.default.gold_fato_scada ALTER COLUMN blade_pitch_a_deg COMMENT 'Ângulo de pitch da pá A (graus, convenção circular 0-360)';
ALTER TABLE workspace.default.gold_fato_scada ALTER COLUMN front_bearing_temp_c COMMENT 'Temperatura do mancal dianteiro (°C)';
ALTER TABLE workspace.default.gold_fato_scada ALTER COLUMN rear_bearing_temp_c COMMENT 'Temperatura do mancal traseiro (°C)';
ALTER TABLE workspace.default.gold_fato_scada ALTER COLUMN gear_oil_temp_c COMMENT 'Temperatura do óleo da multiplicadora (°C)';
ALTER TABLE workspace.default.gold_fato_scada ALTER COLUMN generator_bearing_front_temp_c COMMENT 'Temperatura do mancal dianteiro do gerador (°C)';
ALTER TABLE workspace.default.gold_fato_scada ALTER COLUMN nacelle_ambient_temp_c COMMENT 'Temperatura ambiente na nacele (°C), referência';

-- ===== DIM TURBINA =====
ALTER TABLE workspace.default.gold_dim_turbina ALTER COLUMN turbina_id COMMENT 'PK. Identificador da turbina (1-6)';
ALTER TABLE workspace.default.gold_dim_turbina ALTER COLUMN nome COMMENT 'Nome da turbina';
ALTER TABLE workspace.default.gold_dim_turbina ALTER COLUMN latitude COMMENT 'Latitude (graus)';
ALTER TABLE workspace.default.gold_dim_turbina ALTER COLUMN longitude COMMENT 'Longitude (graus)';
ALTER TABLE workspace.default.gold_dim_turbina ALTER COLUMN potencia_nominal_kw COMMENT 'Potência nominal (kW)';
ALTER TABLE workspace.default.gold_dim_turbina ALTER COLUMN diametro_rotor_m COMMENT 'Diâmetro do rotor (m)';
ALTER TABLE workspace.default.gold_dim_turbina ALTER COLUMN altura_cubo_m COMMENT 'Altura do cubo (m): 68,5 nas turbinas 3 e 6; 78,5 nas demais';

-- ===== DIM TEMPO =====
ALTER TABLE workspace.default.gold_dim_tempo ALTER COLUMN tempo_id COMMENT 'PK. Instante de 10 min (UTC)';
ALTER TABLE workspace.default.gold_dim_tempo ALTER COLUMN data COMMENT 'Data sem hora';
ALTER TABLE workspace.default.gold_dim_tempo ALTER COLUMN ano COMMENT 'Ano';
ALTER TABLE workspace.default.gold_dim_tempo ALTER COLUMN mes COMMENT 'Mês (1-12)';
ALTER TABLE workspace.default.gold_dim_tempo ALTER COLUMN dia COMMENT 'Dia do mês';
ALTER TABLE workspace.default.gold_dim_tempo ALTER COLUMN hora COMMENT 'Hora (0-23)';
ALTER TABLE workspace.default.gold_dim_tempo ALTER COLUMN minuto COMMENT 'Minuto (0, 10, 20, 30, 40, 50)';
ALTER TABLE workspace.default.gold_dim_tempo ALTER COLUMN dia_semana COMMENT 'Dia da semana (em inglês)';

-- ===== DIM STATUS =====
ALTER TABLE workspace.default.gold_dim_status ALTER COLUMN status_id COMMENT 'PK. 0 = Normal, 1 = Stop';
ALTER TABLE workspace.default.gold_dim_status ALTER COLUMN status COMMENT 'Nome do status';
ALTER TABLE workspace.default.gold_dim_status ALTER COLUMN descricao COMMENT 'Descrição do status';